# Open-FDD Engine Tutorial: RTU11 Temperature Sensors

This notebook shows how to run **engine-only FDD** (no Docker, no API) using the **PyPI package**:

```bash
pip install --upgrade open-fdd
```

You will use:
- a real CSV (`examples/engine_iot_playground/data/RTU11.csv`)
- Open-FDD-compatible YAML rules (`examples/engine_iot_playground/rules/*.yaml`)
- sensor-to-rule column mapping

We focus on **temperature sensors first** and compute:
- row-level fault flags
- hourly fault counts / rates
- monthly fault counts / rates

In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import os

import pandas as pd
import yaml

from open_fdd.engine.runner import RuleRunner

# Run notebook from repo root: jupyter lab
REPO_ROOT = Path.cwd()
PLAYGROUND = REPO_ROOT / "examples" / "engine_iot_playground"
RULES_PATH = PLAYGROUND / "rules"
CSV_PATH = PLAYGROUND / "data" / "RTU11.csv"

# Optional override: export OFDD_ENGINE_CSV=/absolute/path/to/RTU11.csv
CSV_PATH = Path(os.environ.get("OFDD_ENGINE_CSV", str(CSV_PATH)))

try:
    print("open-fdd version:", version("open-fdd"))
except PackageNotFoundError:
    print("open-fdd version: not found (run: pip install --upgrade open-fdd)")

print("CSV:", CSV_PATH)
print("Rules:", RULES_PATH)
print("CSV exists:", CSV_PATH.exists())

## 1) Load RTU data and inspect available sensors

Start by loading the CSV and checking column names. This tells you what you can map into YAML rule inputs.

In [ ]:
df = pd.read_csv(CSV_PATH)

# RTU file uses local-time text like: 01-Jan-26 12:00:00 AM EST
# Parse explicitly, then sort chronologically.
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%d-%b-%y %I:%M:%S %p EST",
    errors="coerce",
)
df = df.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)

print(f"Rows loaded: {len(df)}")
print("\nColumns:")
for c in df.columns:
    print("-", c)

df.head(3)

## 2) Read YAML rules and see required input columns

In Open-FDD, mapping is driven by each rule's `inputs.*.column` value.

For this playground, rules currently expect `supply_air_temp_f`.
We will map your RTU sensor column into that expected name.

In [ ]:
def load_rules_preview(rules_dir: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(rules_dir.glob("*.yaml")):
        rule = yaml.safe_load(p.read_text()) or {}
        rule_name = rule.get("name") or rule.get("flag") or p.stem
        inputs = rule.get("inputs") or {}
        for input_name, cfg in inputs.items():
            rows.append(
                {
                    "rule_file": p.name,
                    "rule_name": rule_name,
                    "rule_type": rule.get("type"),
                    "input_key": input_name,
                    "expected_column": (cfg or {}).get("column"),
                    "brick_class": (cfg or {}).get("brick"),
                }
            )
    return pd.DataFrame(rows)

rules_preview = load_rules_preview(RULES_PATH)
rules_preview

## 3) Map RTU temperature sensors -> YAML rule columns

### Temperature sensor-by-sensor notes (RTU11)

- `RTU_11_DA_T(°F)` = discharge air temperature (**used in rules now**)
- `RTU_11_MA_T(°F)` = mixed air temperature (available for future rules)
- `RTU_11_OA_T(°F)` = outside air temperature (available for future rules)
- `RTU_11_RA_T(°F)` = return air temperature (available for future rules)
- `RTU_11_AVGZN_T(°F)` = average zone temperature (available for future rules)

For the included minimal rules, map only discharge air temp to `supply_air_temp_f`.

In [ ]:
column_map = {
    "RTU_11_DA_T(°F)": "supply_air_temp_f",
}

pd.DataFrame(
    [{"source_sensor": k, "rule_column": v} for k, v in column_map.items()]
)

## 4) Run engine-only FDD with RuleRunner

`skip_missing_columns=True` lets you keep a broader CSV while only using mapped inputs for current rules.

In [ ]:
runner = RuleRunner(rules_path=RULES_PATH)

out = runner.run(
    df,
    timestamp_col="Timestamp",
    column_map=column_map,
    skip_missing_columns=True,
    params={"units": "imperial"},
)

flag_cols = [c for c in out.columns if c.endswith("_flag")]
print("Flag columns:", flag_cols)
out[["Timestamp", "RTU_11_DA_T(°F)"] + flag_cols].tail(10)

## 5) Analytical fault metrics: hourly and monthly

The cells below compute fault counts and fault rate (`fault_count / sample_count`) by hour and month.

You can use the same pattern for any future flags.

In [ ]:
summary = out[["Timestamp"] + flag_cols].copy()
summary["hour"] = summary["Timestamp"].dt.floor("h")
summary["month"] = summary["Timestamp"].dt.to_period("M").astype(str)

hourly = (
    summary.groupby("hour")[flag_cols]
    .agg(["sum", "count"])
)

# Flatten multi-index columns and add rates
hourly.columns = [f"{flag}_{metric}" for flag, metric in hourly.columns]
for flag in flag_cols:
    hourly[f"{flag}_rate"] = hourly[f"{flag}_sum"] / hourly[f"{flag}_count"]

monthly = (
    summary.groupby("month")[flag_cols]
    .agg(["sum", "count"])
)
monthly.columns = [f"{flag}_{metric}" for flag, metric in monthly.columns]
for flag in flag_cols:
    monthly[f"{flag}_rate"] = monthly[f"{flag}_sum"] / monthly[f"{flag}_count"]

print("Hourly metrics (tail):")
hourly.tail(10)

In [ ]:
print("Monthly metrics:")
monthly

## 6) Next step: add more temperature rules

To extend beyond discharge-air checks, add YAML rules that reference additional RTU11 temperature sensors (via `column_map`), for example:

- mixed-air plausibility (`RTU_11_MA_T(°F)`)
- outside-air vs mixed-air consistency (`RTU_11_OA_T(°F)` + `RTU_11_MA_T(°F)`)
- return-air drift (`RTU_11_RA_T(°F)`)

Then re-run the same analytics cells to get hourly/monthly fault behavior for each new flag.